# EchoState: does steering a concept change what a model reports about itself?

**The question.** If we alter a language model's internal state, does its self-report change to match?

**The method.** For each model we:

1. Build a *concept direction* contrastively — `mean(activations of positive prompts) − mean(activations of negative prompts)` — so the vector actually encodes something rather than being noise.
2. Inject it into the residual stream at the model's midpoint layer, using a PyTorch forward hook, during every generated token.
3. Compare against a **magnitude-matched random direction**. Both arms push equally hard, so a difference between them is attributable to *direction*, not to force.
4. Measure three things: how far the internals moved (`activation_divergence`), how far the text moved (`output_divergence`), and whether the model *said* it was in the steered state (`introspection_success`).

Strength is expressed as a multiple of each model's own activation magnitude, because residual norms differ by an order of magnitude across models.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis import (
    load_results, steered, effect_by_kind, by_model,
    concept_vs_random_gap, dose_response, introspection_gap,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)

df = load_results("../output/sweep.csv")
print(f"{len(df)} rows, {df['model_name'].nunique()} models")
df.head(3)

## 1. Did steering do anything at all?

Sanity check before interpreting anything: activation divergence should rise with strength, and should be zero for controls.

In [ ]:
effect_by_kind(df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for kind, group in steered(df).groupby("steering_kind"):
    means = group.groupby("steering_strength")["activation_divergence"].mean()
    axes[0].plot(means.index, means.values, marker="o", label=kind)
    means = group.groupby("steering_strength")["output_divergence"].mean()
    axes[1].plot(means.index, means.values, marker="o", label=kind)

axes[0].set(xlabel="steering strength", ylabel="activation divergence",
            title="Internal change")
axes[1].set(xlabel="steering strength", ylabel="output divergence",
            title="Behavioural change")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()

## 2. Is the concept direction special, or is any push equivalent?

This is the control that makes the whole experiment interpretable. If a random direction of the same magnitude moves the output just as much, then the concept direction is not carrying meaning.

In [ ]:
gap = concept_vs_random_gap(df)
gap.sort_values("gap", ascending=False)

## 3. Per-model results

Six models across three architecture families, so the finding is not an artefact of one model's quirks.

In [ ]:
by_model(df)

In [ ]:
summary = by_model(df).set_index("model_name")
ax = summary[["baseline_introspection_rate", "steered_introspection_rate"]].plot(
    kind="barh", figsize=(9, 4)
)
ax.set(xlabel="introspection success rate", ylabel="",
       title="Does steering change what the model reports?")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()

## 4. The actual research question

`changed_output_but_not_report` is the key column: the fraction of runs where steering visibly changed the model's *behaviour* while the model did **not** report being in the steered state.

A high value means the self-report is not tracking the internal state — the model is demonstrably altered and does not say so.

In [ ]:
introspection_gap(df)

## 5. Reading the actual text

Aggregate numbers hide what steering does qualitatively. These are the completions themselves.

In [ ]:
sample = df[(df["model_name"] == df["model_name"].iloc[0])
            & (df["concept_name"] == "positive_affect")]
for _, row in sample.iterrows():
    label = "CONTROL" if row["is_control"] else f"{row['steering_kind']} @ {row['steering_strength']}"
    print(f"[{label:<18}] {row['raw_completion'].strip()[:150]}")
    print()

## Limitations

Stated plainly, because they bound what can be claimed:

- **`introspection_success` detects vocabulary, not understanding.** It asks whether concept words appear in the completion. Words echoed from the prompt are excluded, which removes the largest confound, but the metric still cannot distinguish a genuine self-report from incidental word use. An LLM judge would be the next improvement.
- **These models are small (82M–500M).** None has a meaningful self-model, so a negative introspection result is close to the expected outcome and should not be read as evidence about larger models.
- **The concept directions are built from four prompt pairs each.** That is enough to isolate a direction, not enough to claim it is *the* representation of the concept.
- **Divergence is cosine distance on mean-pooled activations**, which discards positional information and is conservative in absolute terms. Relative ordering is meaningful; the raw magnitude is not.